In [15]:
import pandas as pd
import numpy as np
import surprise
# import matplotlib.pyplot as plt
# import seaborn as sns

In [40]:
df = pd.read_csv("../filmtrust/ratings.txt", sep=' ', names=["uid","iid","rating"])
df

,uid,iid,rating
0,1,1,2.0
1,1,2,4.0
2,1,3,3.5
3,1,4,3.0
4,1,5,4.0
...,...,...,...
35492,1508,84,3.5
35493,1508,17,4.0
35494,1508,669,1.0
35495,1508,686,2.5


In [14]:
lowest_rating = df["rating"].min()
highest_rating = df["rating"].max()
lowest_rating, highest_rating

(np.float64(0.5), np.float64(4.0))

In [16]:
reader = surprise.Reader(rating_scale=(lowest_rating,highest_rating))
data = surprise.Dataset.load_from_df(df, reader)
type(data)

surprise.dataset.DatasetAutoFolds

In [17]:
data

In [18]:
similarity_options = {'name':'cosine','user_based':True}
algo = surprise.KNNBasic(sim_options = similarity_options) # default k=40
output = algo.fit(data.build_full_trainset())

Computing the cosine similarity matrix...
Done computing similarity matrix.


In [19]:
output

## Train Test Split

In [22]:
from surprise.model_selection import train_test_split

In [23]:
trainset, testset = train_test_split(data, test_size=0.25, random_state=26)

print(f"Number of training ratings: {trainset.n_ratings}")
print(f"Number of testing ratings: {len(testset)}")

Number of training ratings: 26622
Number of testing ratings: 8875


In [24]:
similarity_options = {'name':'cosine','user_based':True}
algo = surprise.KNNBasic(sim_options = similarity_options, k=70) # default k=40
output = algo.fit(trainset)
pred = algo.test(testset)
surprise.accuracy.rmse(pred)

Computing the cosine similarity matrix...
Done computing similarity matrix.
RMSE: 0.9282


np.float64(0.9282056050129103)

## Tuning Best K

In [25]:
from surprise.model_selection import GridSearchCV
from surprise.model_selection.split import KFold

In [26]:
params = { 'K':[25,40,50,65,75,85],
           'user_based':[False, True]}
kfold = KFold(n_splits=5, random_state=26, shuffle=True)
gs = GridSearchCV(surprise.KNNBasic, param_grid=params, measures=['rmse','mae'], cv=kfold)
gs.fit(data)

Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computi

In [28]:
print("Best Params : ", gs.best_params['rmse'])
print("Best Score : ", gs.best_score['rmse'])

Best Params :  {'K': 25, 'user_based': False}
Best Score :  0.8653628061278518


In [29]:
print("Best Params : ", gs.best_params['mae'])
print("Best Score : ", gs.best_score['mae'])

Best Params :  {'K': 25, 'user_based': False}
Best Score :  0.6560013418154191


In [30]:
# Building bets model on whole data
algo =gs.best_estimator['rmse']
algo.fit(data.build_full_trainset())

Computing the msd similarity matrix...
Done computing similarity matrix.


## Total Number of Items

In [37]:
iids = df["iid"].unique()
iids

array([   1,    2,    3, ..., 2069, 2070, 2071], shape=(2071,))

## Consider any user

In [33]:
a = [2,4,5,7,8,9]
b = [4,7,9]
np.setdiff1d(a,b)

array([2, 5, 8])

In [47]:
user = 50
u_iid = df[df['uid']==user]['iid'].unique()
print("List of items rated by user : \n",u_iid)
print("\nNo of items rated by user {0} : {1} ".format(user, len(u_iid)))
iids_ro_pred = np.setdiff1d(iids, u_iid)
print("\nItems not rated by the user or those items for which the expected ratings are to be predicted : ", iids_ro_pred)

List of items rated by user : 
 [  8 211   3   2 219 234  12 254 250 207  11 253 236  84  10   7 233  13
   1   5   6 252 241 216 257 206   4 217   9 215 213  17 255 220 121 245
 239 251 235]

No of items rated by user 50 : 39 

Items not rated by the user or those items for which the expected ratings are to be predicted :  [  14   15   16 ... 2069 2070 2071]


In [47]:
user = 50
u_iid = df[df['uid']==user]['iid'].unique()
print("List of items rated by user : \n",u_iid)
print("\nNo of items rated by user {0} : {1} ".format(user, len(u_iid)))
iids_ro_pred = np.setdiff1d(iids, u_iid)
print("\nItems not rated by the user or those items for which the expected ratings are to be predicted : ", iids_ro_pred)

List of items rated by user : 
 [  8 211   3   2 219 234  12 254 250 207  11 253 236  84  10   7 233  13
   1   5   6 252 241 216 257 206   4 217   9 215 213  17 255 220 121 245
 239 251 235]

No of items rated by user 50 : 39 

Items not rated by the user or those items for which the expected ratings are to be predicted :  [  14   15   16 ... 2069 2070 2071]


In [52]:
testset = [[user,iid,0.] for iid in iids_ro_pred]
predictions = algo.test(testset)

exp_ratings = [(predictions[i].iid,predictions[i].est) for i in range(0,len(predictions))]
exp_ratings = pd.DataFrame(exp_ratings,columns=['iid','est_rating'])
exp_ratings.sort_values(by='est_rating',ascending=False).head(30)

,iid,est_rating
1096,1136,4.0
1978,2018,4.0
1986,2026,4.0
1974,2014,4.0
1973,2013,4.0
2024,2064,4.0
786,826,4.0
787,827,4.0
1948,1988,4.0
1937,1977,4.0
